In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import seaborn as sns
import scipy.stats as stats

In [2]:
print("Loading datasets...")

DATA_DIR = "../../data/raw/"
df_orders      = pd.read_csv(DATA_DIR + "orders.csv", parse_dates=["order_date"])
df_order_items = pd.read_csv(DATA_DIR + "order_items.csv", low_memory=False)
df_products    = pd.read_csv(DATA_DIR + "products.csv")
df_promotions  = pd.read_csv(DATA_DIR + "promotions.csv", parse_dates=["start_date", "end_date"])
df_customers   = pd.read_csv(DATA_DIR + "customers.csv", parse_dates=["signup_date"])
df_geography   = pd.read_csv(DATA_DIR + "geography.csv")
df_payments    = pd.read_csv(DATA_DIR + "payments.csv")
df_returns     = pd.read_csv(DATA_DIR + "returns.csv", parse_dates=["return_date"])
df_reviews     = pd.read_csv(DATA_DIR + "reviews.csv", parse_dates=["review_date"])
df_shipments   = pd.read_csv(DATA_DIR + "shipments.csv", parse_dates=["ship_date", "delivery_date"])
df_inventory   = pd.read_csv(DATA_DIR + "inventory.csv", parse_dates=["snapshot_date"])
df_web_traffic = pd.read_csv(DATA_DIR + "web_traffic.csv", parse_dates=["date"])
df_sales       = pd.read_csv(DATA_DIR + "sales.csv", parse_dates=["Date"])

print("All datasets loaded.\n")

Loading datasets...
All datasets loaded.



In [46]:
def build_features(df):
    df_trans = df.copy()
    
    # Date & Time features
    df_time = df.copy()
    df_trans['year'] = df_trans['Date'].dt.year
    df_trans['month'] = df_trans['Date'].dt.month           
    df_trans['day_of_month'] = df_trans['Date'].dt.day      
    df_trans['day_of_week'] = df_trans['Date'].dt.dayofweek 
    df_trans['day_of_year'] = df_trans['Date'].dt.dayofyear
    
    df_trans['is_weekend'] = df_trans['day_of_week'].isin([5, 6]).astype(int) # Saturdays, Sundays
    df_trans['is_month_end'] = (df_trans['day_of_month'] >= 27).astype(int)

    df_trans['is_even_year'] = (df_trans['year'] % 2 == 0).astype(int)
    df_trans['is_2019_crash'] = (df_trans['year'] == 2019).astype(int)
    
    # Double day
    df_trans['is_double_day'] = ((df_trans['month'] == df_trans['day_of_month']) & (df_trans['month'] >= 9)).astype(int)
    df_trans['current_double_day'] = pd.to_datetime(df_trans['year'].astype(str) + '-' + 
                                                    df_trans['month'].astype(str) + '-' + 
                                                    df_trans['month'].astype(str), errors='coerce')
    df_trans['days_to_sale'] = (df_trans['current_double_day'] - df_trans['Date']).dt.days
    df_trans['days_until_double_day'] = df_trans['days_to_sale'].apply(lambda x: x if 0 < x <= 5 else 0)
    df_trans.drop(columns=['current_double_day', 'days_to_sale'], inplace=True)
    
    # Holiday event
    df_trans['is_holiday'] = (
        ((df_trans['day_of_month'] == 30) & (df_trans['month'] == 4)) |  # 30/4
        ((df_trans['day_of_month'] == 1) & (df_trans['month'] == 5)) |   # 1/5
        ((df_trans['day_of_month'] == 2) & (df_trans['month'] == 9)) |   # 2/9
        ((df_trans['day_of_month'] == 1) & (df_trans['month'] == 1)) |   # Tet
        ((df_trans['day_of_month'] == 24) & (df_trans['month'] == 12))   # Noel
    ).astype(int)

    # Fourier 
    df_trans['month_sin'] = np.sin(2 * np.pi * df_trans['month'] / 12)
    df_trans['month_cos'] = np.cos(2 * np.pi * df_trans['month'] / 12)
    df_trans['day_year_sin'] = np.sin(2 * np.pi * df_trans['day_of_year'] / 365.25)
    df_trans['day_year_cos'] = np.cos(2 * np.pi * df_trans['day_of_year'] / 365.25)

    # Lag features
    df_trans['lag_364'] = df_trans['Revenue'].shift(364)
    df_trans['lag_365'] = df_trans['Revenue'].shift(365)
    df_trans['lag_728'] = df_trans['Revenue'].shift(728)
    df_trans['sessions_lag_364'] = df_trans['sessions'].shift(364) # web traffic
    
    # Rolling Window
    df_trans['rolling_7_lag_364'] = df_trans['lag_364'].rolling(window=7, min_periods=1).mean()
    df_trans['rolling_30_lag_364'] = df_trans['lag_364'].rolling(window=30, min_periods=1).mean()
    df_trans['rolling_7_sessions_lag_364'] = df_trans['sessions_lag_364'].rolling(window=7, min_periods=1).mean()
    
    # Historical Promo Probability
    train_mask = df_trans['year'] < 2023
    promo_matrix = df_trans[train_mask].groupby(
        ['month', 'day_of_month', 'day_of_week']
    )['has_promo'].mean().reset_index()
    promo_matrix.rename(columns={'has_promo': 'hist_promo_prob'}, inplace=True)
    df_trans = pd.merge(df_trans, promo_matrix, on=['month', 'day_of_month', 'day_of_week'], how='left')
    df_trans['hist_promo_prob'] = df_trans['hist_promo_prob'].fillna(0)

    
    return df_trans

In [47]:
df_tmp = df_sales.copy()
df_tmp['has_promo'] = 0

for _, row in df_promotions.iterrows():
    mask = (
        (df_tmp['Date'] >= row['start_date']) &
        (df_tmp['Date'] <= row['end_date'])
    )
    df_tmp.loc[mask, 'has_promo'] = 1

df_web_traffic_copy = df_web_traffic.copy()
df_web_traffic_copy = df_web_traffic_copy.rename(columns={'date': 'Date'})
df_tmp = df_tmp.merge(df_web_traffic_copy[['Date', 'sessions']], on='Date')

In [50]:
test_dates = pd.date_range(start='2023-01-01', end='2024-07-01', freq='D')
df_test = pd.DataFrame({'Date': test_dates})
df_full = pd.concat([df_tmp, df_test], ignore_index=True)

df = build_features(df_full)
df.tail(31)

,Date,Revenue,COGS,has_promo,sessions,year,month,day_of_month,day_of_week,day_of_year,...,day_year_cos,lag_364,lag_365,lag_728,sessions_lag_364,rolling_7_lag_364,rolling_30_lag_364,rolling_7_sessions_lag_364,hist_promo_prob,is_holiday
4169,2024-06-01,NaN,NaN,NaN,NaN,2024,6,1,5,153,...,-0.872929,NaN,NaN,2998952.42,NaN,NaN,NaN,NaN,0.0,0
4170,2024-06-02,NaN,NaN,NaN,NaN,2024,6,2,6,154,...,-0.881192,NaN,NaN,2945501.79,NaN,NaN,NaN,NaN,0.0,0
4171,2024-06-03,NaN,NaN,NaN,NaN,2024,6,3,0,155,...,-0.889193,NaN,NaN,3100859.00,NaN,NaN,NaN,NaN,0.0,0
4172,2024-06-04,NaN,NaN,NaN,NaN,2024,6,4,1,156,...,-0.896932,NaN,NaN,3673238.80,NaN,NaN,NaN,NaN,0.0,0
4173,2024-06-05,NaN,NaN,NaN,NaN,2024,6,5,2,157,...,-0.904405,NaN,NaN,3686799.90,NaN,NaN,NaN,NaN,0.0,0
4174,2024-06-06,NaN,NaN,NaN,NaN,2024,6,6,3,158,...,-0.911611,NaN,NaN,2619038.14,NaN,NaN,NaN,NaN,0.0,0
4175,2024-06-07,NaN,NaN,NaN,NaN,2024,6,7,4,159,...,-0.918547,NaN,NaN,3369544.01,NaN,NaN,NaN,NaN,0.0,0
4176,2024-06-08,NaN,NaN,NaN,NaN,2024,6,8,5,160,...,-0.925211,NaN,NaN,4463584.56,NaN,NaN,NaN,NaN,0.0,0
4177,2024-06-09,NaN,NaN,NaN,NaN,2024,6,9,6,161,...,-0.931601,NaN,NaN,4561669.81,NaN,NaN,NaN,NaN,0.0,0
4178,2024-06-10,NaN,NaN,NaN,NaN,2024,6,10,0,162,...,-0.937716,NaN,NaN,4221162.05,NaN,NaN,NaN,NaN,0.0,0


In [52]:
df.columns

Index(['Date', 'Revenue', 'COGS', 'has_promo', 'sessions', 'year', 'month',
       'day_of_month', 'day_of_week', 'day_of_year', 'is_weekend',
       'is_month_end', 'is_even_year', 'is_2019_crash', 'is_double_day',
       'days_until_double_day', 'month_sin', 'month_cos', 'day_year_sin',
       'day_year_cos', 'lag_364', 'lag_365', 'lag_728', 'sessions_lag_364',
       'rolling_7_lag_364', 'rolling_30_lag_364', 'rolling_7_sessions_lag_364',
       'hist_promo_prob', 'is_holiday'],
      dtype='object')

In [60]:
features = [
    'day_of_week', 
    'is_weekend', 
    'is_month_end', 
    'is_even_year', 
    'is_2019_crash', 
    'is_double_day', 
    'days_until_double_day', 
    'is_holiday',
    
    'month_sin', 
    'month_cos', 
    'day_year_sin', 
    'day_year_cos',
    
    'lag_364', 
    'lag_365', 
    'lag_728', 
    'rolling_7_lag_364', 
    'rolling_30_lag_364', 
    
    'sessions_lag_364', 
    'rolling_7_sessions_lag_364', 
    'hist_promo_prob'
]
df_trans = df.copy()
df_trans.head()

,Date,Revenue,COGS,has_promo,sessions,year,month,day_of_month,day_of_week,day_of_year,...,day_year_cos,lag_364,lag_365,lag_728,sessions_lag_364,rolling_7_lag_364,rolling_30_lag_364,rolling_7_sessions_lag_364,hist_promo_prob,is_holiday
0,2013-01-01,5304546.99,4156070.20,0.0,9760.0,2013,1,1,1,1,...,0.999852,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.5,1
1,2013-01-02,1606940.44,1237497.84,0.0,10456.0,2013,1,2,2,2,...,0.999408,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.5,0
2,2013-01-03,2281680.01,1832133.02,0.0,10076.0,2013,1,3,3,3,...,0.998669,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
3,2013-01-04,2376895.46,1940747.07,0.0,9973.0,2013,1,4,4,4,...,0.997634,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
4,2013-01-05,2509462.77,1977027.71,0.0,10223.0,2013,1,5,5,5,...,0.996303,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0


## Train Model

In [57]:
#!pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 563.4 kB/s  0:00:03 eta 0:00:01

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

features = [
    'day_of_week', 'is_weekend', 'is_month_end', 'is_even_year', 
    'is_2019_crash', 'is_double_day', 'days_until_double_day', 'is_holiday',
    'month_sin', 'month_cos', 'day_year_sin', 'day_year_cos',
    'lag_364', 'lag_365', 'lag_728', 
    'rolling_7_lag_364', 'rolling_30_lag_364', 
    'sessions_lag_364', 'rolling_7_sessions_lag_364', 'hist_promo_prob'
]
target = 'Revenue'

VAL_START_DATE = '2021-07-01'
TEST_START_DATE = '2023-01-01'

train_df = df_trans[df_trans['Date'] < VAL_START_DATE].copy()
val_df = df_trans[(df_trans['Date'] >= VAL_START_DATE) & (df_trans['Date'] < TEST_START_DATE)].copy()

X_train, y_train = train_df[features], train_df[target]
X_val, y_val = val_df[features], val_df[target]

lgb_train = lgb.Dataset(X_train, y_train)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

print("Track 1: Main Model (Tweedie)")
params_main = {
    'objective': 'tweedie',
    'tweedie_variance_power': 1.5, 
    'metric': 'rmse',
    'learning_rate': 0.03,        
    'num_leaves': 31,              
    'feature_fraction': 0.8,      
    'bagging_fraction': 0.8,     
    'bagging_freq': 1,
    'seed': 42,
    'verbose': -1
}

model_main = lgb.train(
    params_main, 
    lgb_train, 
    num_boost_round=1500, 
    valid_sets=[lgb_train, lgb_val], 
    callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
)
val_df['pred_main'] = model_main.predict(X_val)

In [62]:
val_df

,Date,Revenue,COGS,has_promo,sessions,year,month,day_of_month,day_of_week,day_of_year,...,lag_364,lag_365,lag_728,sessions_lag_364,rolling_7_lag_364,rolling_30_lag_364,rolling_7_sessions_lag_364,hist_promo_prob,is_holiday,pred_main
3103,2021-07-01,4975867.15,5024857.74,1.0,37571.0,2021,7,1,3,182,...,4866951.38,4785799.65,1489388.03,27460.0,4.415276e+06,3.650934e+06,36792.714286,1.0,0,4.408231e+06
3104,2021-07-02,4397949.68,4326458.07,1.0,33233.0,2021,7,2,4,183,...,3696972.88,4866951.38,1841931.90,30149.0,4.437849e+06,3.657595e+06,35304.142857,1.0,0,4.547383e+06
3105,2021-07-03,3132513.38,3101143.37,1.0,35152.0,2021,7,3,5,184,...,706367.78,3696972.88,1673667.75,36231.0,4.117043e+06,3.582392e+06,35150.000000,1.0,0,3.939234e+06
3106,2021-07-04,730601.15,647312.84,1.0,32810.0,2021,7,4,6,185,...,1406074.50,706367.78,2108446.49,32159.0,3.741747e+06,3.547518e+06,34116.285714,1.0,0,2.764822e+06
3107,2021-07-05,1878817.85,1927429.12,1.0,29990.0,2021,7,5,0,186,...,2077312.99,1406074.50,2671803.54,31155.0,3.292308e+06,3.541476e+06,32847.000000,1.0,0,2.759434e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3647,2022-12-27,2100553.66,2184872.24,1.0,17416.0,2022,12,27,1,361,...,1855064.19,1025037.51,2940732.69,21237.0,1.644396e+06,1.475791e+06,19971.428571,1.0,0,1.544852e+06
3648,2022-12-28,3448729.20,3513621.00,1.0,21071.0,2022,12,28,2,362,...,4065184.90,1855064.19,2881135.64,22058.0,1.892602e+06,1.550269e+06,20376.000000,1.0,0,2.120360e+06
3649,2022-12-29,3083944.33,3170787.10,1.0,20884.0,2022,12,29,3,363,...,3842586.50,4065184.90,2153864.87,19928.0,2.096413e+06,1.578757e+06,20350.285714,1.0,0,3.202419e+06
3650,2022-12-30,2884668.76,3022292.15,1.0,17679.0,2022,12,30,4,364,...,3041203.79,3842586.50,1553431.51,18742.0,2.323897e+06,1.617411e+06,20489.571429,1.0,0,3.056233e+06


In [ ]:
def print_metrics(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"[{name}] MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | R²: {r2:.4f}")

print_metrics(val_df[target], val_df['pred_main'], "LGBM Main")